[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/fold_from_contacts1.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/Open-Athena/MarinFold/blob/main/notebooks/fold_from_contacts1.ipynb)

# MarinFold — Fold From Contacts: MarinFold vs MSA

A tiny, classical **"approximate AlphaFold"** pipeline — predict a residue–residue
**contact map**, turn it into a 3D backbone by distance geometry, and compare to the
known structure. It follows Sergey Ovchinnikov's
[`AlphaFold_approx_v2`](https://colab.research.google.com/github/sokrypton/ml4me/blob/main/AlphaFold_approx_v2.ipynb)
notebook, with **one change**: the contacts can come from our current best
**contacts-v1 1.5B** model — predicted **from the single query sequence, no MSA** —
instead of from MSA coevolution.

We still load the MSA and compute the classic **coevolution** contacts, so you can:

- **see both contact maps side by side** (MarinFold vs MSA vs ground truth), and
- **toggle `CONTACT_SOURCE`** to choose which one drives the folded 3D structure —
  then fold *both* and compare their RMSD to the reference.

The reference structure and the MSA both come bundled from the **AlphaFold DB** entry
for a UniProt id, so a single id gives us everything: a sequence, an MSA, and a
structure to score against.

## How it works

**Contacts → 3D** (the shared distance-geometry core, from the reference notebook):

1. Pick the top contacts from a score matrix (either source), plus fixed backbone
   priors for near-neighbours (consecutive Cα ≈ 3.8 Å).
2. **Floyd–Warshall** fills in every missing pairwise distance via the shortest path
   (triangle inequality).
3. **Classical MDS** eigen-decomposes the completed distance matrix into 3D coordinates.
4. Rescale, fix chirality (MDS can return a mirror image), and Kabsch-align to the
   reference for the RMSD and the overlay.

**The two contact sources:**

- **MSA coevolution** — one-hot the alignment, invert the covariance (a GREMLIN-style
  precision matrix), APC-correct. Needs a deep MSA; no learned model.
- **MarinFold contacts-v1** — the packaged `marinfold` model reads `P(contact)` for
  every residue pair **from the query sequence alone** (the `pairwise` readout from
  exp82/exp89). No MSA, no template, no structure.

**Runtime selection is automatic**, matched to the hardware (identical to the
[Inference Example 1](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/inference_example1.ipynb)
notebook):

- Ampere+ NVIDIA GPU (compute ≥ 8.0) → MarinFold `vllm` (CUDA-12 wheel).
- Older NVIDIA GPU (Colab/Kaggle **free T4**, compute < 8.0) → MarinFold `transformers`, on the GPU.
- Apple Silicon → `mlx`; CPU / TPU-host → `transformers`.

The **default Colab free T4** runs the `transformers` GPU path and finishes in seconds.
After the install cell Colab may need **Runtime → Restart session** before `import
marinfold` works; if so, restart and rerun from the import cell.

**Ground truth here is a Cα contact** (Cα–Cα < 8 Å, sequence separation ≥ 6) read off
the AlphaFold model — a standard, self-contained definition for this demo. It is *not*
the pyconfind side-chain definition contacts-v1 was trained on, so treat the GT panel
as an approximate reference, not the training target.

In [ ]:
# @title Configuration
UNIPROT_ID = "P0A8I3"  # @param {type:"string"}
CONTACT_SOURCE = "marinfold"  # @param ["marinfold", "msa"]
CONTACTS_PER_RESIDUE = 2.0  # @param {type:"number"}

# --- MarinFold contacts-v1 knobs (used when the model is run) ---
MODEL_NICKNAME = "contacts-v1-exp75-1.5B"  # @param ["contacts-v1-exp75-1.5B"]
METHOD = "pairwise"  # @param ["pairwise", "rollout"]
ENSEMBLE_K = 1  # @param {type:"integer"}
N_ROLLOUTS = 100  # @param {type:"integer"}
BACKEND_MODE = "auto"  # @param ["auto", "vllm", "mlx", "transformers"]
DTYPE = "bfloat16"  # @param ["bfloat16", "float16", "float32"]
BATCH_SIZE = 32  # @param {type:"integer"}
GPU_MEMORY_UTILIZATION = 0.8  # @param {type:"number"}

# CONTACT_SOURCE selects which contact map is folded into the 3D structure below.
# We always compute BOTH (MSA coevolution + MarinFold contacts-v1) so the maps can
# be compared, and a later cell folds both to compare RMSD regardless of this toggle.
# CONTACTS_PER_RESIDUE * L is how many top-ranked pairs seed the distance geometry.

print({
    "uniprot_id": UNIPROT_ID.upper(),
    "contact_source": CONTACT_SOURCE,
    "contacts_per_residue": CONTACTS_PER_RESIDUE,
    "model": MODEL_NICKNAME,
    "method": METHOD,
    "ensemble_k": ENSEMBLE_K,
    "n_rollouts": N_ROLLOUTS,
    "backend_mode": BACKEND_MODE,
    "dtype": DTYPE,
    "batch_size": BATCH_SIZE,
})

In [ ]:
# @title Clone the repo and pick a runtime
import os
import platform
import shutil
import subprocess
from pathlib import Path


def _find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "marinfold" / "pyproject.toml").exists():
            return candidate
    return None


def _has_nvidia_gpu() -> bool:
    if shutil.which("nvidia-smi") is None:
        return False
    result = subprocess.run(["nvidia-smi", "-L"], stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True, check=False)
    return result.returncode == 0 and bool(result.stdout.strip())


def _nvidia_compute_capability():
    """Highest NVIDIA compute capability present (e.g. 7.5 for a T4), or None.

    vLLM's current wheels only run on Ampere or newer (>= 8.0); Turing/Pascal
    cards (the free T4/P100) must use transformers, so this gates the choice.
    """
    if shutil.which("nvidia-smi") is None:
        return None
    result = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False)
    if result.returncode != 0:
        return None
    caps = []
    for line in result.stdout.splitlines():
        try:
            caps.append(float(line.strip()))
        except ValueError:
            continue
    return max(caps) if caps else None


def _has_tpu() -> bool:
    return any(os.environ.get(k) for k in ("COLAB_TPU_ADDR", "TPU_NAME", "TPU_WORKER_ID"))


def _is_apple_silicon() -> bool:
    return platform.system() == "Darwin" and platform.machine() == "arm64"


def _auto_backend_choice():
    if _has_nvidia_gpu():
        cap = _nvidia_compute_capability()
        if cap is not None and cap >= 8.0:
            return {"backend": "vllm", "extra": "vllm", "runtime": f"nvidia-gpu-sm{cap:g}",
                    "note": f"Using vLLM: NVIDIA GPU compute capability {cap:g} (>= 8.0)."}
        cap_str = f"{cap:g}" if cap is not None else "unknown"
        return {"backend": "transformers", "extra": "transformers",
                "runtime": f"nvidia-gpu-sm{cap_str}",
                "note": (f"NVIDIA GPU compute capability {cap_str} (< 8.0, e.g. a Colab/"
                         "Kaggle T4) -> running on the GPU via transformers.")}
    if _has_tpu():
        return {"backend": "transformers", "extra": "transformers", "runtime": "tpu-host-cpu",
                "note": ("TPU runtime -> running on the TPU VM's HOST CPU via transformers "
                         "(not the TPU chips). Use a GPU runtime or Kaggle for speed.")}
    if _is_apple_silicon():
        return {"backend": "mlx", "extra": "mlx", "runtime": "apple-silicon",
                "note": "Using MLX because Apple Silicon is available."}
    return {"backend": "transformers", "extra": "transformers", "runtime": "cpu-or-other",
            "note": "Falling back to transformers (no supported accelerator detected)."}


BASE_WORKDIR = Path("/content") if Path("/content").exists() and os.access("/content", os.W_OK) else Path.cwd()
REPO_DIR = _find_repo_root(Path.cwd()) or (BASE_WORKDIR / "MarinFold")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Open-Athena/MarinFold.git", str(REPO_DIR)], check=True)

AUTO_BACKEND = _auto_backend_choice()
if BACKEND_MODE == "auto":
    BACKEND_NAME = AUTO_BACKEND["backend"]
    EXTRA_NAME = AUTO_BACKEND["extra"]
else:
    BACKEND_NAME = EXTRA_NAME = BACKEND_MODE

print(f"repo checkout: {REPO_DIR}")
print({"backend": BACKEND_NAME, "extra": EXTRA_NAME,
       "runtime": AUTO_BACKEND["runtime"], "note": AUTO_BACKEND["note"]})

In [ ]:
# @title Install marinfold + notebook deps
import importlib.util
import shutil
import subprocess
import sys

PACKAGE_DIR = REPO_DIR / "marinfold"

if importlib.util.find_spec("pip") is not None:
    _PIP = [sys.executable, "-m", "pip", "install", "-q"]
elif shutil.which("uv") is not None:
    _PIP = ["uv", "pip", "install", "--python", sys.executable, "-q"]
else:
    raise RuntimeError("Neither pip nor uv is available to install notebook dependencies.")

if EXTRA_NAME == "vllm":
    # Colab drivers are CUDA 12.x but vLLM's default PyPI wheel is now CUDA 13
    # (`ImportError: libcudart.so.13`). Pull a pinned cu129 wheel from vLLM's index
    # (matches the driver). Also install transformers as a runtime fallback backend,
    # in ONE command so pip resolves transformers once against both constraints.
    VLLM_VERSION = "0.20.2"
    EXTRAS = "transformers"
    install_args = ["-e", f".[{EXTRAS}]", f"vllm=={VLLM_VERSION}+cu129",
                    "--extra-index-url", f"https://wheels.vllm.ai/{VLLM_VERSION}/cu129/",
                    "--extra-index-url", "https://download.pytorch.org/whl/cu129"]
else:
    EXTRAS = EXTRA_NAME
    install_args = ["-e", f".[{EXTRAS}]"]

subprocess.run(_PIP + install_args, cwd=PACKAGE_DIR, check=True)
# py3Dmol for the interactive 3D overlay (no MSA-tool / pyconfind needed: the GT here
# is a Ca-distance contact read straight off the structure).
subprocess.run(_PIP + ["py3Dmol"], check=True)

# Make the freshly-installed editable package importable without a restart.
if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))
importlib.invalidate_caches()

_label = f"{EXTRAS} + vllm=={VLLM_VERSION}(cu129)" if EXTRA_NAME == "vllm" else EXTRAS
print(f"installed marinfold extras: {_label}  (+ py3Dmol)")
print("If the next cell raises ModuleNotFoundError, do Runtime -> Restart session "
      "and rerun from the import cell (skip clone/install).")

In [ ]:
# @title Imports + runtime summary
import importlib
import platform
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import py3Dmol

from marinfold import list_model_entries, resolve_model_entry
from marinfold.document_structures.contacts_v1 import (
    InferenceConfig,
    predict,
    structure_from_sequence,
)

os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")

runtime_summary = {"backend": BACKEND_NAME, "runtime": AUTO_BACKEND["runtime"],
                   "platform": platform.platform(), "machine": platform.machine()}
if importlib.util.find_spec("torch") is not None:
    import torch
    runtime_summary["torch_version"] = torch.__version__
    runtime_summary["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        runtime_summary["gpu_name"] = props.name

entry = resolve_model_entry(MODEL_NICKNAME)
print("Available models:", sorted(e.nickname for e in list_model_entries()))
print(f"Using model: {entry.nickname}  ({entry.url})")
print(runtime_summary)

In [ ]:
# @title Geometry, coevolution, and viewer helpers (from the reference notebook)
# get_ca / xyz_to_dm / get_coevolution / floyd_warshall / classic_mds / parse_a3m /
# seq_to_num are ported from sokrypton/ml4me AlphaFold_approx_v2. Kabsch alignment
# and the py3Dmol Ca-trace viewer are added for the comparison here.


def get_ca(path):
    """(L, 3) Ca coordinates parsed from a PDB file, in residue order."""
    xyz = []
    with open(path) as fh:
        for line in fh:
            if line[:4] == "ATOM" and line[12:16].strip() == "CA":
                xyz.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.array(xyz)


def xyz_to_dm(X):
    return np.sqrt(np.sum(np.square(X[None, :] - X[:, None]), axis=-1))


def parse_a3m(path):
    """Parse an a3m; drop lowercase (insertion) columns. Returns list of sequences."""
    seqs, seq, have = [], "", False
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line.startswith(">"):
                if have:
                    seqs.append("".join(c for c in seq if c.isupper() or c == "-"))
                seq, have = "", True
            else:
                seq += line
    if have:
        seqs.append("".join(c for c in seq if c.isupper() or c == "-"))
    return seqs


def seq_to_num(sequences):
    aa = "ACDEFGHIKLMNPQRSTVWY-"
    idx = {a: i for i, a in enumerate(aa)}
    return np.array([[idx.get(c, 20) for c in s] for s in sequences])


def get_coevolution(X):
    """GREMLIN-style coevolution (inverse-covariance + APC) contact scores."""
    Y = np.eye(21)[X]
    N, L, A = Y.shape
    c = np.cov(Y.reshape(N, -1).T)
    ic = np.linalg.inv(c + 4.5 / np.sqrt(N) * np.eye(c.shape[0]))
    d = np.diag(ic)
    pcc = ic / np.sqrt(d[:, None] * d[None, :])
    raw = np.sqrt(np.square(pcc.reshape(L, A, L, A)[:, :20, :, :20]).sum((1, 3)))
    np.fill_diagonal(raw, 0)
    ap = raw.sum(0, keepdims=True) * raw.sum(1, keepdims=True) / raw.sum()
    apc = raw - ap
    np.fill_diagonal(apc, 0)
    return apc


def floyd_warshall(d):
    for m in range(d.shape[0]):
        o = d[m]
        d = np.minimum(d, o[None, :] + o[:, None])
    return d


def classic_mds(M):
    L = M.shape[0]
    d = np.square(M)
    c = np.eye(L) - np.ones((L, L)) / L
    m = -0.5 * c @ d @ c
    s, u = np.linalg.eigh(m)
    return u[:, -3:] * np.sqrt(np.clip(s[-3:], 0, None))


# ---- contacts -> 3D ----

def select_top_contacts(score, n_contacts, min_sep=4):
    """Boolean (L, L) mask of the top-`n_contacts` pairs (i<j, |i-j|>=min_sep)."""
    L = score.shape[0]
    s = np.array(score, dtype=np.float64)
    s[~np.isfinite(s)] = -np.inf
    iu = np.triu_indices(L, k=min_sep)
    vals = s[iu]
    finite = int(np.isfinite(vals).sum())
    n = max(0, min(int(n_contacts), finite))
    mask = np.zeros((L, L), dtype=bool)
    if n:
        keep = np.argsort(-vals)[:n]
        ii, jj = iu[0][keep], iu[1][keep]
        mask[ii, jj] = mask[jj, ii] = True
    return mask


def fold_from_contacts(score, n_contacts):
    """Contact score matrix -> (L, 3) coords via sparse distances + FW + MDS."""
    L = score.shape[0]
    idx = np.arange(L)
    rel = np.abs(idx[:, None] - idx[None, :])
    mask = select_top_contacts(score, n_contacts)
    dm = np.where(mask, 7.0, np.inf)      # a predicted contact ~ 7 A apart
    dm[rel == 0] = 0.0
    dm[rel == 1] = 3.8                    # consecutive Ca
    dm[rel == 2] = 6.8
    dm[rel == 3] = 7.0
    coords = classic_mds(floyd_warshall(dm))
    # Rescale so consecutive-Ca spacing is ~3.8 A.
    step = np.median(np.sqrt(np.square(np.diff(coords, axis=0)).sum(1)))
    if step > 0:
        coords = coords * (3.8 / step)
    return coords, mask


# ---- Kabsch alignment (with optional reflection for MDS mirror images) ----

def _kabsch_rot(P, Q, allow_reflection):
    H = P.T @ Q
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if not allow_reflection and np.linalg.det(R) < 0:
        Vt = Vt.copy()
        Vt[-1] *= -1
        R = Vt.T @ U.T
    return R


def align_a_to_b(a, b, allow_reflection=False):
    """Rigid-align `a` onto `b`; returns (aligned_a, rmsd). Tries the mirror image
    too when allow_reflection (classical MDS can yield a reflected structure)."""
    ca, cb = a.mean(0), b.mean(0)
    A, B = a - ca, b - cb
    best, best_rmsd = None, np.inf
    for mirror in ((1, 1, 1), (1, 1, -1)) if allow_reflection else ((1, 1, 1),):
        Am = A * np.array(mirror)
        aligned = Am @ _kabsch_rot(Am, B, allow_reflection=False).T
        rmsd = float(np.sqrt(np.square(aligned - B).sum(1).mean()))
        if rmsd < best_rmsd:
            best, best_rmsd = aligned + cb, rmsd
    return best, best_rmsd


# ---- py3Dmol Ca-trace viewer ----

def _ca_pdb(xyz, chain="A"):
    lines = []
    for i, (x, y, z) in enumerate(xyz, start=1):
        lines.append(
            f"ATOM  {i:>5} {' CA ':<4} GLY {chain}{i:>4}    "
            f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           C"
        )
    for i in range(1, len(xyz)):
        lines.append(f"CONECT{i:>5}{i + 1:>5}")
    lines.append("END")
    return "\n".join(lines)


def show_ca_traces(traces, width=640, height=480):
    """traces: list of (xyz, hex_color, label). Renders connected Ca traces."""
    view = py3Dmol.view(width=width, height=height)
    for xyz, color, _ in traces:
        view.addModel(_ca_pdb(xyz), "pdb")
        view.setStyle({"model": -1}, {"stick": {"color": color, "radius": 0.3},
                                       "sphere": {"color": color, "scale": 0.28}})
    view.zoomTo()
    print("legend:", ", ".join(f"{lbl}={col}" for _, col, lbl in traces))
    return view


print("helpers ready")

In [ ]:
# @title Download the AlphaFold structure + MSA, build the query sequence
uid = UNIPROT_ID.upper()
RUN_DIR = BASE_WORKDIR / "fold_from_contacts_runs"
RUN_DIR.mkdir(parents=True, exist_ok=True)

pdb_path = RUN_DIR / f"AF-{uid}-F1-model_v6.pdb"
a3m_path = RUN_DIR / f"AF-{uid}-F1-msa_v6.a3m"
pdb_url = f"https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v6.pdb"
a3m_url = f"https://alphafold.ebi.ac.uk/files/msa/AF-{uid}-F1-msa_v6.a3m"
if not pdb_path.exists():
    urlretrieve(pdb_url, pdb_path)
if not a3m_path.exists():
    urlretrieve(a3m_url, a3m_path)

# Optional: reduce MSA redundancy with sokrypton's filter (better coevolution).
# Falls back to the raw MSA if the C++ tool can't be built.
filt_path = RUN_DIR / f"AF-{uid}-F1-msa_v6.filt.a3m"
filter_bin = RUN_DIR / "filter"
try:
    if not filter_bin.exists():
        cpp = RUN_DIR / "filter.cpp"
        urlretrieve("https://raw.githubusercontent.com/sokrypton/msa-tools/refs/heads/main/filter.cpp", cpp)
        subprocess.run(["g++", "-O3", "-std=c++17", "-fopenmp", str(cpp), "-o", str(filter_bin)], check=True)
    subprocess.run([str(filter_bin), str(a3m_path), str(filt_path),
                    "-id", "90", "-cov", "75", "-qid", "15"], check=True)
    msa_source = filt_path
except Exception as exc:  # noqa: BLE001 - filtering is a nice-to-have
    print(f"[warn] MSA filtering unavailable ({type(exc).__name__}: {exc}); using the raw MSA.")
    msa_source = a3m_path

sequences = parse_a3m(msa_source)
query_seq = sequences[0].replace("-", "")            # first row = the query, no gaps
msa = seq_to_num(sequences)
xyz_ref = get_ca(pdb_path)                            # reference Ca coordinates
L = len(query_seq)

# The MSA columns, the model sequence, and the structure must be the same length.
if not (msa.shape[1] == L == len(xyz_ref)):
    raise ValueError(
        f"length mismatch for {uid}: query={L}, msa_cols={msa.shape[1]}, "
        f"structure_ca={len(xyz_ref)}. This demo expects a full-length (F1) "
        f"AlphaFold entry whose MSA query row matches the model; try another UniProt id."
    )

print(f"{uid}: L={L} residues, {msa.shape[0]} MSA sequences ({msa_source.name})")
print(f"query: {query_seq[:60]}{'...' if L > 60 else ''}")

In [ ]:
# @title MSA coevolution contacts
con_msa = get_coevolution(msa)
print(f"MSA coevolution contact matrix: {con_msa.shape}")

In [ ]:
# @title MarinFold contacts-v1 contacts (from the query sequence alone)
from dataclasses import replace

cfg = InferenceConfig(
    model=MODEL_NICKNAME, backend=BACKEND_NAME, method=METHOD,
    n_rollouts=N_ROLLOUTS, ensemble_k=ENSEMBLE_K, batch_size=BATCH_SIZE,
    dtype=DTYPE, gpu_memory_utilization=GPU_MEMORY_UTILIZATION, keep_matrix=True,
)


def _predict_matrix(config):
    """Run predict() for the query sequence; retry on transformers if a non-
    transformers backend can't load/run (e.g. a vLLM CUDA/arch mismatch)."""
    structs = [structure_from_sequence(query_seq, entry_id=uid)]
    try:
        rec = next(iter(predict(config, structures=structs)))
    except Exception as exc:  # noqa: BLE001 - demo: degrade instead of crashing
        if config.backend == "transformers":
            raise
        print(f"[warn] backend {config.backend!r} failed ({type(exc).__name__}: {exc}); "
              "retrying on transformers.")
        config = replace(config, backend="transformers")
        rec = next(iter(predict(config, structures=structs)))
    return config, rec


cfg, record = _predict_matrix(cfg)
BACKEND_NAME = cfg.backend  # reflect any fallback
con_marin = np.array(record["score_matrix"], dtype=np.float64)  # (L, L), NaN on the near band
print(f"MarinFold contact matrix: {con_marin.shape} (backend={cfg.backend}, method={cfg.method})")

In [ ]:
# @title Compare the contact maps: ground truth vs MSA vs MarinFold
idx = np.arange(L)
rel = np.abs(idx[:, None] - idx[None, :])
MIN_SEP = 6

# Ground-truth Ca contacts (Ca-Ca < 8 A, |i-j| >= 6) off the reference structure.
dm_ref = xyz_to_dm(xyz_ref)
gt = (dm_ref < 8.0) & (rel >= MIN_SEP)
n_gt = int(gt.sum() // 2)


def _precision_topL(score, k):
    """Precision of the top-k predicted pairs (|i-j|>=6) against the GT contacts."""
    s = np.array(score, dtype=np.float64)
    s[~np.isfinite(s)] = -np.inf
    s[rel < MIN_SEP] = -np.inf
    iu = np.triu_indices(L, k=MIN_SEP)
    order = np.argsort(-s[iu])[:k]
    hit = gt[iu[0][order], iu[1][order]]
    return float(hit.mean()) if len(order) else float("nan")


for name, score in [("MSA coevolution", con_msa), ("MarinFold cv1", con_marin)]:
    print(f"{name:>16}: precision@L = {_precision_topL(score, L):.3f}, "
          f"precision@L/5 = {_precision_topL(score, max(1, L // 5)):.3f}")

# Mask the meaningless near-diagonal band for display.
def _disp(score):
    d = np.array(score, dtype=np.float64)
    d[rel < MIN_SEP] = np.nan
    return d

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(np.where(gt, 1.0, np.nan), cmap="Greys", origin="lower", vmin=0, vmax=1)
axes[0].set_title(f"ground truth (Ca<8A)\n{n_gt} contacts")
for ax, name, score, cmap in [
    (axes[1], "MSA coevolution", con_msa, "viridis"),
    (axes[2], f"MarinFold-cv1 ({cfg.method})", con_marin, "magma"),
]:
    d = _disp(score)
    vmax = np.nanpercentile(d, 99.5)
    ax.imshow(d, cmap=cmap, origin="lower", vmin=np.nanmin(d), vmax=vmax)
    ax.set_title(f"{name}\nfrom {'MSA' if 'MSA' in name else 'sequence alone'}")
for ax in axes:
    ax.set_xlabel("residue j")
    ax.set_ylabel("residue i")
fig.suptitle(f"{uid}  |  L={L}", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# @title Fold the selected contact source into a 3D backbone
n_contacts = int(round(CONTACTS_PER_RESIDUE * L))
CONTACT_SCORES = {"marinfold": con_marin, "msa": con_msa}
selected_score = CONTACT_SCORES[CONTACT_SOURCE]

coords_pred, mask = fold_from_contacts(selected_score, n_contacts)
coords_aligned, rmsd = align_a_to_b(coords_pred, xyz_ref, allow_reflection=True)

dm_pred = xyz_to_dm(coords_aligned)
print(f"CONTACT_SOURCE = {CONTACT_SOURCE!r}  |  {n_contacts} seeded contacts")
print(f"Ca-RMSD to the AlphaFold reference: {rmsd:.2f} A")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
axes[0].imshow(dm_ref, origin="lower"); axes[0].set_title("reference distance map")
axes[1].imshow(dm_pred, origin="lower"); axes[1].set_title(f"reconstructed ({CONTACT_SOURCE})")
axes[2].scatter(dm_ref.ravel(), dm_pred.ravel(), s=1, alpha=0.05)
lim = max(dm_ref.max(), dm_pred.max())
axes[2].plot([0, lim], [0, lim], "r-", lw=0.8)
axes[2].set_xlabel("reference distance (A)"); axes[2].set_ylabel("reconstructed distance (A)")
axes[2].set_title("pairwise distances")
fig.tight_layout()
plt.show()

In [ ]:
# @title Fold BOTH sources and compare RMSD
results = {}
for source, score in CONTACT_SCORES.items():
    coords, _ = fold_from_contacts(score, n_contacts)
    aligned, r = align_a_to_b(coords, xyz_ref, allow_reflection=True)
    results[source] = {"coords": aligned, "rmsd": r}

for source, res in sorted(results.items(), key=lambda kv: kv[1]["rmsd"]):
    star = "  <- selected" if source == CONTACT_SOURCE else ""
    print(f"{source:>10}: Ca-RMSD = {res['rmsd']:.2f} A{star}")

fig, ax = plt.subplots(figsize=(4.5, 3.5))
srcs = list(results)
ax.bar(srcs, [results[s]["rmsd"] for s in srcs],
       color=["#d62728" if s == CONTACT_SOURCE else "#1f77b4" for s in srcs])
ax.set_ylabel("Ca-RMSD to reference (A)")
ax.set_title(f"{uid}: folding accuracy by contact source")
for i, s in enumerate(srcs):
    ax.text(i, results[s]["rmsd"], f"{results[s]['rmsd']:.1f}", ha="center", va="bottom")
fig.tight_layout()
plt.show()

In [ ]:
# @title Interactive 3D overlay: reference vs reconstruction
# Grey = AlphaFold reference. Colored = the reconstruction from CONTACT_SOURCE.
# Rotate/zoom with the mouse. Switch CONTACT_SOURCE above and rerun to compare.
show_ca_traces([
    (xyz_ref, "#888888", "reference (AlphaFold)"),
    (results[CONTACT_SOURCE]["coords"], "#d62728", f"{CONTACT_SOURCE} recon"),
]).show()

## Notes & knobs

- **`CONTACT_SOURCE`** toggles which map is folded in the single-source cell and
  highlighted in the overlay. The "fold both" cell always reports both RMSDs, so you
  can compare MarinFold-from-sequence against MSA-coevolution head-to-head.
- **`CONTACTS_PER_RESIDUE`** sets how many top-ranked pairs seed the distance geometry
  (`× L`). Too few underconstrains the fold; too many admits false contacts. ~1–3 is a
  reasonable range.
- **`METHOD="rollout"` / `ENSEMBLE_K>1`** are MarinFold-side accuracy knobs (slower);
  `pairwise` with `ENSEMBLE_K=1` is the fast default.
- **Deeper MSAs help coevolution**, not MarinFold. On shallow-MSA / orphan sequences
  the sequence-only MarinFold contacts are expected to hold up better — a good case to
  try by picking a UniProt id with few homologs.
- This is a deliberately **classical, minimal** structure module (Floyd–Warshall + MDS),
  not a learned folder — expect coarse backbones, not AlphaFold-grade models. The point
  is to compare the two *contact* signals through the same simple lens.